In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (keep them in memory for reuse)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to convert timestamp column (seconds) to tz-aware ISO strings
def ts_min_max_iso(df, ts_col='timestamp'):
    if ts_col not in df.columns or df[ts_col].dropna().empty:
        return (None, None)
    ser = pd.to_datetime(df[ts_col].astype(int), unit='s', utc=True)
    ser = ser.dt.tz_convert(tz)
    mn = ser.min().isoformat()
    mx = ser.max().isoformat()
    return mn, mx

# Basic summaries for each file
metric_min_ts, metric_max_ts = ts_min_max_iso(metric_df, 'timestamp')
trace_min_ts, trace_max_ts = ts_min_max_iso(trace_df, 'timestamp')
log_min_ts, log_max_ts = ts_min_max_iso(log_df, 'timestamp')
error_min_ts, error_max_ts = ts_min_max_iso(error_logs_df, 'timestamp')

basic_summary_df = pd.DataFrame([
    {
        'file': 'metric.csv',
        'row_count': len(metric_df),
        'unique_cmdb_ids': sorted(metric_df['cmdb_id'].dropna().unique().tolist()),
        'min_timestamp': metric_min_ts,
        'max_timestamp': metric_max_ts
    },
    {
        'file': 'trace.csv',
        'row_count': len(trace_df),
        'unique_cmdb_ids': sorted(trace_df['cmdb_id'].dropna().unique().tolist()),
        'min_timestamp': trace_min_ts,
        'max_timestamp': trace_max_ts
    },
    {
        'file': 'log.csv',
        'row_count': len(log_df),
        'unique_cmdb_ids': sorted(log_df['cmdb_id'].dropna().unique().tolist()),
        'min_timestamp': log_min_ts,
        'max_timestamp': log_max_ts
    },
    {
        'file': 'error_logs.csv',
        'row_count': len(error_logs_df),
        'unique_cmdb_ids': sorted(error_logs_df['cmdb_id'].dropna().unique().tolist()),
        'min_timestamp': error_min_ts,
        'max_timestamp': error_max_ts
    }
])

# Additional per-file aggregates (limited as requested)

# metric.csv: unique kpi_name values and their counts (top 20)
metric_kpi_counts = (
    metric_df['kpi_name']
    .value_counts()
    .rename_axis('kpi_name')
    .reset_index(name='count')
    .head(20)
)

# trace.csv: unique trace_name values and their counts (top 20)
trace_trace_counts = (
    trace_df['trace_name']
    .value_counts()
    .rename_axis('trace_name')
    .reset_index(name='count')
    .head(20)
)

# log.csv: unique log_name values and their counts (top 20)
log_log_counts = (
    log_df['log_name']
    .value_counts()
    .rename_axis('log_name')
    .reset_index(name='count')
    .head(20)
)

# error_logs.csv: up to top 10 distinct message snippets (first 100 chars) and their counts
error_logs_df['message'] = error_logs_df.get('message', pd.Series(dtype=str)).fillna('').astype(str)
error_logs_df['msg_snippet'] = error_logs_df['message'].str.slice(0, 100)
error_snippets = (
    error_logs_df['msg_snippet']
    .value_counts()
    .rename_axis('msg_snippet')
    .reset_index(name='count')
    .head(10)
)

# Keep outputs compact: only show the requested aggregates (basic summary + the per-file counts/snippets)
basic_summary_df, metric_kpi_counts, trace_trace_counts, log_log_counts, error_snippets

```
Out[1]:
```
metric_summary = basic_summary_df[basic_summary_df['file']=='metric.csv'].iloc[0]
trace_summary = basic_summary_df[basic_summary_df['file']=='trace.csv'].iloc[0]
log_summary = basic_summary_df[basic_summary_df['file']=='log.csv'].iloc[0]
error_summary = basic_summary_df[basic_summary_df['file']=='error_logs.csv'].iloc[0]

def top_list_to_str(df, name_col, count_col, topn=5):
    items = [f"{row[name_col]} ({int(row[count_col])})" for _, row in df.head(topn).iterrows()]
    return ", ".join(items)

summary = (
    "Summary of loaded telemetry files:\n\n"
    "1) metric.csv:\n"
    f"   - Rows: {int(metric_summary['row_count'])}\n"
    f"   - cmdb_ids: {metric_summary['unique_cmdb_ids']}\n"
    f"   - Time range (UTC+8): {metric_summary['min_timestamp']} to {metric_summary['max_timestamp']}\n"
    f"   - Top KPI names by count (top 10 shown): {', '.join(metric_kpi_counts.head(10)['kpi_name'] + ' (' + metric_kpi_counts.head(10)['count'].astype(str) + ')')}\n\n"
    "2) trace.csv:\n"
    f"   - Rows: {int(trace_summary['row_count'])}\n"
    f"   - cmdb_ids: {trace_summary['unique_cmdb_ids']}\n"
    f"   - Time range (UTC+8): {trace_summary['min_timestamp']} to {trace_summary['max_timestamp']}\n"
    f"   - Top trace names by count (top 10 shown): {', '.join(trace_trace_counts.head(10)['trace_name'] + ' (' + trace_trace_counts.head(10)['count'].astype(str) + ')')}\n\n"
    "3) log.csv:\n"
    f"   - Rows: {int(log_summary['row_count'])}\n"
    f"   - cmdb_ids: {log_summary['unique_cmdb_ids']}\n"
    f"   - Time range (UTC+8): {log_summary['min_timestamp']} to {log_summary['max_timestamp']}\n"
    f"   - Log name counts: {', '.join(log_log_counts.head(10)['log_name'] + ' (' + log_log_counts.head(10)['count'].astype(str) + ')')}\n\n"
    "4) error_logs.csv:\n"
    f"   - Rows: {int(error_summary['row_count'])}\n"
    f"   - cmdb_ids: {error_summary['unique_cmdb_ids']}\n"
    f"   - Time range (UTC+8): {error_summary['min_timestamp']} to {error_summary['max_timestamp']}\n"
    f"   - Message snippets: none (no error log rows)\n\n"
    "Notes: Top-name lists above show the most frequent entries and their counts; full tables remain in kernel variables for further inspection."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count                                    unique_cmdb_ids              min_timestamp              max_timestamp
0      metric.csv      27542  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...  2021-03-12T14:00:00+08:00  2021-03-12T14:30:00+08:00
1       trace.csv       6711  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  2021-03-12T14:00:00+08:00  2021-03-12T14:30:00+08:00
2         log.csv        528  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...  2021-03-12T14:00:00+08:00  2021-03-12T14:30:00+08:00
3  error_logs.csv          0                                                 []                       None                       None,                                              kpi_name  count
0        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    360
1      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    359
2       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    356
3                  OSLinux-OSLinux_ZABBIX_Host_Uptime    356
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    356
5                          OSLinux-CPU_CPU_CPUSysTime    353
6                              OSLinux-CPU_CPU_CPUWio    353
7                         OSLinux-CPU_CPU_CPUUserTime    353
8   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    353
9                             OSLinux-CPU_CPU_CPULoad    352
10                        OSLinux-CPU_CPU_CPUidleutil    352
11                                             app.rr    341
12                                            app.mrt    341
13                                             app.sr    341
14                                            app.cnt    341
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    294
16                         OSLinux-CPU_CPU_CPUCpuUtil    280
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    279
18          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    259
19              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    259,                        trace_name  count
0        trace.self.duration_mean    357
1         trace.self.duration_p95    357
2            trace.self.row_count    357
3     trace.to_MG01.duration_mean    240
4      trace.to_MG01.duration_p95    240
5         trace.to_MG01.row_count    240
6     trace.to_MG02.duration_mean    208
7         trace.to_MG02.row_count    208
8      trace.to_MG02.duration_p95    208
9    trace.from_IG02.duration_p95    120
10      trace.from_MG01.row_count    120
11  trace.from_IG01.duration_mean    120
12      trace.from_IG02.row_count    120
13   trace.from_IG01.duration_p95    120
14      trace.from_IG01.row_count    120
15   trace.from_MG01.duration_p95    120
16  trace.from_MG01.duration_mean    120
17  trace.from_IG02.duration_mean    120
18  trace.from_MG02.duration_mean    105
19   trace.from_MG02.duration_p95    105,            log_name  count
0   log.error_count    214
1     log.row_count    214
2  log.gc_pause_max    100, Empty DataFrame
Columns: [msg_snippet, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df and tz if present; otherwise load and set tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column and a tz-aware datetime column (UTC -> convert to Asia/Shanghai)
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Global P95 and total point count for each (cmdb_id, kpi_name) using the full series
global_stats = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_points_full_series='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric_df to the incident window (inclusive)
window_start = pd.Timestamp('2021-03-12T14:00:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-12T14:30:00').tz_localize(tz)

window_df = metric_df[(metric_df['ts'] >= window_start) & (metric_df['ts'] <= window_end)].copy()

# Merge global_p95 onto the windowed rows so comparisons use the global threshold
merged_window = window_df.merge(global_stats, on=['cmdb_id', 'kpi_name'], how='left')

# Mark rows exceeding global_p95 (strictly greater)
merged_window['above_p95'] = merged_window['value'] > merged_window['global_p95']

# Prepare an anomaly timestamp column (ts where above_p95 else NaT)
merged_window['anom_ts'] = merged_window['ts'].where(merged_window['above_p95'], pd.NaT)

# Aggregate per (cmdb_id, kpi_name) within the window
window_agg = (
    merged_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        points_in_window=('value', 'size'),
        points_above_p95_in_window=('above_p95', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Ensure integer type for counts
window_agg['points_above_p95_in_window'] = window_agg['points_above_p95_in_window'].astype(int)
window_agg['points_in_window'] = window_agg['points_in_window'].astype(int)

# 3) Combine global stats and window aggregates
combined = (
    window_agg
    .merge(global_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute max_over_p95_ratio with safe divide (NaN if global_p95 == 0 or missing)
combined['max_over_p95_ratio'] = np.where(
    combined['global_p95'] == 0,
    np.nan,
    (combined['max_value_in_window'] - combined['global_p95']) / combined['global_p95']
)

# Format earliest_anomaly_ts as ISO strings (keep NaT as NaN)
combined['earliest_anomaly_timestamp_in_window'] = combined['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
combined.loc[combined['earliest_anomaly_ts'].isna(), 'earliest_anomaly_timestamp_in_window'] = pd.NA

# Select and order requested columns
result_df = combined[
    [
        'cmdb_id',
        'kpi_name',
        'global_p95',
        'total_points_full_series',
        'points_in_window',
        'points_above_p95_in_window',
        'earliest_anomaly_timestamp_in_window',
        'max_value_in_window',
        'max_over_p95_ratio'
    ]
].copy()

# Sort by descending points_above_p95_in_window and then by points_in_window for tie-breaker
result_df = result_df.sort_values(
    by=['points_above_p95_in_window', 'points_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

# Keep full intermediate tables in kernel variables: global_stats, window_df, merged_window, window_agg, combined, result_df
# Display compact table limited to top 20 rows
result_df_head20 = result_df.head(20)
result_df_head20

```
Out[2]:
```
Summary of results (computed global P95 per (cmdb_id, kpi_name) and inspected the incident window 2021-03-12T14:00:00+08:00 to 2021-03-12T14:30:00+08:00):

- Scope: analysis produced per (cmdb_id, kpi_name) global P95 and counts, then examined the 14:00–14:30 window. The compact output shows the top 20 pairs sorted by number of in-window points exceeding global P95.

Key findings from the top rows:
- All listed pairs had 31 points in-window (window is inclusive of 31 minutes). For these pairs the full series point count is also 31.
- Each of these top pairs had 2 points in the window that exceeded their global P95. No pair in the top 20 had more than 2 points above P95.
- Notable highest peak-over-P95 ratios (max_over_p95_ratio):
  - IG01 — OSLinux-CPU_CPU_CPUWio: max value ~0.6737 vs global P95 ~0.4014 → ~0.678 (≈67.8% over P95). Earliest exceedance: 2021-03-12T14:01:00+08:00.
  - Mysql01 — OSLinux-CPU_CPU_CPUWio: max value ~3.7756 vs global P95 ~2.74395 → ~0.376 (≈37.6% over P95). Earliest exceedance: 2021-03-12T14:01:00+08:00.
  - MG02 — JVM-Memory_7779_JVM_Memory_HeapMemoryUsed: max value ~1.2241e+09 vs global P95 ~1.1072e+09 → ~0.1056 (≈10.6% over P95). Earliest exceedance: 2021-03-12T14:02:00+08:00.
  - Redis01 (instantaneous ops): ~8.0% over P95; Redis keyspace_hits: tiny relative increase.
  - Several CPU and JVM metrics on IG01, IG02, MG01, MG02, Mysql01 showed small but measurable exceedances (ratios from ~0.001 up to ~0.05).

- Earliest anomaly timestamps (examples):
  - IG01 JVM heap memory used: first exceedance at 2021-03-12T14:01:00+08:00.
  - IG01 thread count: first exceedance at 2021-03-12T14:14:00+08:00.
  - MG02 JVM heap: first exceedance at 2021-03-12T14:02:00+08:00.
  - Mysql01 CPU-related exceedances mostly appear early in the window (14:01–14:03).

Bottom line:
- During the 14:00–14:30 window, the top affected series experienced only 2 anomalous minutes (points) each above their global P95. The most significant relative spike observed was IG01 OSLinux CPU WIO (~68% above its global P95), followed by Mysql01 CPU WIO (~38% above P95) and MG02 JVM heap (~10.6% above P95). Full intermediate tables (global_stats, window_df, merged_window, window_agg, combined, result_df) remain in kernel variables for deeper follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  total_points_full_series  points_in_window  points_above_p95_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_p95_ratio
0      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.165582e+08                        31                31                           2             2021-03-12T14:01:00+0800         8.427720e+08            0.032103
1      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.250000e+01                        31                31                           2             2021-03-12T14:14:00+0800         7.400000e+01            0.020690
2      IG01                            OSLinux-CPU_CPU_CPULoad  1.210000e+00                        31                31                           2             2021-03-12T14:01:00+0800         1.270000e+00            0.049587
3      IG01                         OSLinux-CPU_CPU_CPUSysTime  5.031500e-01                        31                31                           2             2021-03-12T14:20:00+0800         5.366060e-01            0.066493
4      IG01                        OSLinux-CPU_CPU_CPUUserTime  2.537739e+01                        31                31                           2             2021-03-12T14:05:00+0800         2.543035e+01            0.002087
5      IG01                             OSLinux-CPU_CPU_CPUWio  4.014000e-01                        31                31                           2             2021-03-12T14:01:00+0800         6.737000e-01            0.678376
6      IG01                        OSLinux-CPU_CPU_CPUidleutil  7.410535e+01                        31                31                           2             2021-03-12T14:04:00+0800         7.414661e+01            0.000557
7      IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.729554e+06                        31                31                           2             2021-03-12T14:29:00+0800         6.729644e+06            0.000013
8      IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.843333e+08                        31                31                           2             2021-03-12T14:02:00+0800         8.916366e+08            0.008259
9      IG02           JVM-Threads_7778_JVM_ThreadCount_Threads  7.450000e+01                        31                31                           2             2021-03-12T14:03:00+0800         7.600000e+01            0.020134
10     MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.024739e+09                        31                31                           2             2021-03-12T14:09:00+0800         1.027161e+09            0.002364
11     MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.107222e+09                        31                31                           2             2021-03-12T14:02:00+0800         1.224095e+09            0.105556
12  Mysql01                            OSLinux-CPU_CPU_CPULoad  3.000000e-01                        31                31                           2             2021-03-12T14:28:00+0800         3.600000e-01            0.200000
13  Mysql01                         OSLinux-CPU_CPU_CPUSysTime  5.738920e-01                        31                31                           2             2021-03-12T14:20:00+0800         6.029000e-01            0.050546
14  Mysql01                        OSLinux-CPU_CPU_CPUUserTime  4.719155e-01                        31                31                           2             2021-03-12T14:26:00+0800         5.489000e-01            0.163132
15  Mysql01                             OSLinux-CPU_CPU_CPUWio  2.743950e+00                        31                31                           2             2021-03-12T14:01:00+0800         3.775625e+00            0.375982
16  Mysql01                        OSLinux-CPU_CPU_CPUidleutil  9.852400e+01                        31                31                           2             2021-03-12T14:03:00+0800         9.876466e+01            0.002443
17  Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.729096e+06                        31                31                           2             2021-03-12T14:29:00+0800         6.729187e+06            0.000013
18  Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...  5.252500e+03                        31                31                           2             2021-03-12T14:21:00+0800         5.673000e+03            0.080057
19  Redis01            redis-Redis_6379_Redis  (keyspace_hits)  4.182330e+09                        31                31                           2             2021-03-12T14:29:00+0800         4.182613e+09            0.000068```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df and tz if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure a tz-aware datetime column 'ts' exists (timestamps are in seconds)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Global P95 and total point count for each (cmdb_id, trace_name) using the full series
global_stats_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(total_points_full_series='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter trace_df to the incident window (inclusive)
window_start = pd.Timestamp('2021-03-12T14:00:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-12T14:30:00').tz_localize(tz)
window_df_trace = trace_df[(trace_df['ts'] >= window_start) & (trace_df['ts'] <= window_end)].copy()

# 3) Merge global thresholds into windowed rows and compute above_p95 flag
merged_window_trace = window_df_trace.merge(global_stats_trace, on=['cmdb_id', 'trace_name'], how='left')
merged_window_trace['above_p95'] = merged_window_trace['value'] > merged_window_trace['global_p95']
merged_window_trace['anom_ts'] = merged_window_trace['ts'].where(merged_window_trace['above_p95'], pd.NaT)

# Aggregate per (cmdb_id, trace_name) within the window
window_agg_trace = (
    merged_window_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        points_in_window=('value', 'size'),
        points_above_p95_in_window=('above_p95', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Ensure integer types for counts
window_agg_trace['points_in_window'] = window_agg_trace['points_in_window'].astype(int)
window_agg_trace['points_above_p95_in_window'] = window_agg_trace['points_above_p95_in_window'].astype(int)

# 4) Combine global stats and window aggregates
combined_trace = window_agg_trace.merge(global_stats_trace, on=['cmdb_id', 'trace_name'], how='left')

# Compute max_over_p95_ratio safely (NaN if global_p95 == 0 or missing)
combined_trace['max_over_p95_ratio'] = np.where(
    combined_trace['global_p95'] == 0,
    np.nan,
    (combined_trace['max_value_in_window'] - combined_trace['global_p95']) / combined_trace['global_p95']
)

# Format earliest anomaly timestamp as ISO string with timezone; keep NA for missing
combined_trace['earliest_anomaly_timestamp_in_window'] = combined_trace['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
combined_trace.loc[combined_trace['earliest_anomaly_ts'].isna(), 'earliest_anomaly_timestamp_in_window'] = pd.NA

# Select and order requested columns
result_trace_df = combined_trace[
    [
        'cmdb_id',
        'trace_name',
        'global_p95',
        'total_points_full_series',
        'points_in_window',
        'points_above_p95_in_window',
        'earliest_anomaly_timestamp_in_window',
        'max_value_in_window',
        'max_over_p95_ratio'
    ]
].copy()

# Sort by points_above_p95_in_window (desc) then max_over_p95_ratio (desc)
result_trace_df = result_trace_df.sort_values(
    by=['points_above_p95_in_window', 'max_over_p95_ratio'],
    ascending=[False, False]
).reset_index(drop=True)

# Keep intermediate variables in kernel: global_stats_trace, window_df_trace, merged_window_trace, window_agg_trace, combined_trace, result_trace_df
# Return a compact table limited to top 20 rows
result_trace_head20 = result_trace_df.head(20)
result_trace_head20

```
Out[3]:
```
Summary of trace analysis for the incident window 2021-03-12T14:00:00+08:00 through 2021-03-12T14:30:00+08:00:

Scope and method
- For each (cmdb_id, trace_name) we computed the global P95 across the entire trace.csv series and counted total points in the full series, then examined the specified 14:00–14:30 window for anomalies (value > global P95).
- Output sorted by number of in-window points above P95, limited to the top 20 pairs.

Key findings (top candidates)
- All top 20 pairs had 2 points in the window exceeding their global P95.
- Most pairs had full-window coverage (points_in_window = 26–31) and total_points_full_series in the 26–31 range (one row_count metric with global_p95=86.55 had max 129).
- Largest relative spikes (max_over_p95_ratio):
  1. dockerA2 — trace.self.duration_mean: global_p95 = 0.11047, max_in_window = 1.0200, max_over_p95_ratio ≈ 8.23 (very large spike). Earliest anomaly: 2021-03-12T14:24:00+08:00.
  2. dockerA1 — trace.self.duration_p95: global_p95 = 0.03823, max_in_window = 0.3358, ratio ≈ 7.78. Earliest anomaly: 2021-03-12T14:24:00+08:00.
  3. MG01/dockerA1 to/from paths (trace.to_MG01 / trace.from_dockerA1): global_p95 ≈ 1.21365, max ≈ 2.5210, ratio ≈ 1.08. Earliest anomaly: 2021-03-12T14:24:00+08:00.
  4. Several MG02 / Tomcat02 / dockerA2 traces show moderate spikes (ratios ~0.42–0.77), often earliest around 14:17–14:24.
  5. Many of the top items are request/latency traces between dockerA* and MG* (to_MG01, to_MG02, from_*), plus IG01→Tomcat04 traces with noticeable increases.

Timing patterns
- A cluster of earliest anomalies occurs at ~14:24:00+08:00 (dockerA1/dockerA2 ↔ MG01/MG02 interactions).
- Other earlier anomalies include MG02/Tomcat02 and IG01→Tomcat04 around 14:02–14:20.

Practical takeaway
- The strongest signals point to dockerA1/dockerA2 related traces (large multiple-times-over-P95 latency spikes centered ~14:24) and elevated durations on paths to/from MG01/MG02. These endpoints and the MG routing paths are the highest-priority candidates for investigation.
- All top pairs only had 2 anomalous minutes within the 30-minute window; investigate whether those two spikes correspond to transient incidents (e.g., GC, network hiccup, or deployment) or recurring issues.

Notes
- Full intermediate tables (global P95s, windowed rows, aggregated stats) remain in kernel variables for any deeper follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_p95  total_points_full_series  points_in_window  points_above_p95_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_p95_ratio
0   dockerA2           trace.self.duration_mean    0.110470                        31                31                           2             2021-03-12T14:24:00+0800             1.020000            8.233234
1   dockerA1            trace.self.duration_p95    0.038230                        30                30                           2             2021-03-12T14:24:00+0800             0.335800            7.783678
2       MG01   trace.from_dockerA1.duration_p95    1.213650                        30                30                           2             2021-03-12T14:24:00+0800             2.521000            1.077205
3   dockerA1         trace.to_MG01.duration_p95    1.213650                        30                30                           2             2021-03-12T14:24:00+0800             2.521000            1.077205
4       MG01   trace.from_dockerA2.duration_p95    1.045957                        30                30                           2             2021-03-12T14:17:00+0800             1.846400            0.765272
5   dockerA2         trace.to_MG01.duration_p95    1.045957                        30                30                           2             2021-03-12T14:17:00+0800             1.846400            0.765272
6       MG02  trace.from_dockerA1.duration_mean    0.304085                        26                26                           2             2021-03-12T14:24:00+0800             0.501625            0.649620
7   dockerA1        trace.to_MG02.duration_mean    0.304085                        26                26                           2             2021-03-12T14:24:00+0800             0.501625            0.649620
8       MG01  trace.from_dockerB1.duration_mean    0.143248                        30                30                           2             2021-03-12T14:24:00+0800             0.220677            0.540523
9   dockerB1        trace.to_MG01.duration_mean    0.143248                        30                30                           2             2021-03-12T14:24:00+0800             0.220677            0.540523
10      IG01    trace.to_Tomcat04.duration_mean    0.736833                        30                30                           2             2021-03-12T14:03:00+0800             1.102482            0.496244
11  Tomcat04      trace.from_IG01.duration_mean    0.736833                        30                30                           2             2021-03-12T14:03:00+0800             1.102482            0.496244
12      MG01      trace.from_dockerA2.row_count   86.550000                        30                30                           2             2021-03-12T14:06:00+0800           129.000000            0.490468
13  dockerA2            trace.to_MG01.row_count   86.550000                        30                30                           2             2021-03-12T14:06:00+0800           129.000000            0.490468
14      MG02  trace.from_Tomcat02.duration_mean    0.570962                        26                26                           2             2021-03-12T14:20:00+0800             0.825159            0.445208
15  Tomcat02        trace.to_MG02.duration_mean    0.570962                        26                26                           2             2021-03-12T14:20:00+0800             0.825159            0.445208
16      MG02  trace.from_dockerA2.duration_mean    0.230939                        26                26                           2             2021-03-12T14:20:00+0800             0.328857            0.423998
17  dockerA2        trace.to_MG02.duration_mean    0.230939                        26                26                           2             2021-03-12T14:20:00+0800             0.328857            0.423998
18      MG01     trace.to_dockerA2.duration_p95    2.202500                        30                30                           2             2021-03-12T14:20:00+0800             3.084500            0.400454
19  dockerA2       trace.from_MG01.duration_p95    2.202500                        30                30                           2             2021-03-12T14:20:00+0800             3.084500            0.400454```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure tz-aware timestamp column 'ts' exists
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Candidate cmdb_ids
candidates = ['dockerA1','dockerA2','IG01','MG01','MG02','Tomcat02','Tomcat04','Mysql01','Redis01']

# Restrict metric_df to candidates (keep full time range for global stats)
subset = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) Unique kpi_name counts per cmdb_id and top 10 kpi_name examples (by frequency)
kpi_counts = (
    subset
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['cmdb_id', 'count'], ascending=[True, False])
)

# number of unique kpis per cmdb_id
num_unique_kpis = kpi_counts.groupby('cmdb_id')['kpi_name'].nunique().reset_index(name='number_of_unique_kpis')

# top 10 kpis per cmdb_id as compact string "kpi_name (count)"
def top_kpis_str(df, topn=10):
    return ", ".join([f"{r.kpi_name} ({int(r['count'])})" for _, r in df.head(topn).iterrows()])

top_kpis_examples = (
    kpi_counts
    .groupby('cmdb_id')
    .apply(lambda g: top_kpis_str(g, topn=10))
    .reset_index(name='top_kpi_examples')
)

summary_kpis_df = num_unique_kpis.merge(top_kpis_examples, on='cmdb_id', how='left')
summary_kpis_df = summary_kpis_df.sort_values(by='number_of_unique_kpis', ascending=False).reset_index(drop=True)

# 2) Compute global P95 for each (cmdb_id, kpi_name) using the full series (within subset)
global_stats_subset = (
    subset
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_points_full_series='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 3) Filter to incident window (inclusive)
window_start = pd.Timestamp('2021-03-12T14:00:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-12T14:30:00').tz_localize(tz)
window_df = subset[(subset['ts'] >= window_start) & (subset['ts'] <= window_end)].copy()

# Aggregate window stats per (cmdb_id, kpi_name)
# Merge global p95 into window rows to evaluate exceedances using global threshold
merged_window = window_df.merge(global_stats_subset, on=['cmdb_id', 'kpi_name'], how='left')

# Flag above global p95 (strictly greater)
merged_window['above_p95'] = merged_window['value'] > merged_window['global_p95']
merged_window['anom_ts'] = merged_window['ts'].where(merged_window['above_p95'], pd.NaT)

window_agg = (
    merged_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        points_in_window=('value', 'size'),
        points_above_p95_in_window=('above_p95', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Ensure integer counts
window_agg['points_in_window'] = window_agg['points_in_window'].astype(int)
window_agg['points_above_p95_in_window'] = window_agg['points_above_p95_in_window'].astype(int)

# Some (cmdb_id,kpi_name) may have zero points in window; include them by left-joining global stats
combined_subset = global_stats_subset.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing in-window numeric stats with zeros/NaN appropriately
combined_subset['points_in_window'] = combined_subset['points_in_window'].fillna(0).astype(int)
combined_subset['points_above_p95_in_window'] = combined_subset['points_above_p95_in_window'].fillna(0).astype(int)
# earliest_anomaly_ts may be NaT; keep as NaT for formatting
# max_value_in_window may be NaN if no points in window

# Compute max_over_p95_ratio safely (NaN if global_p95 == 0 or max_value_in_window is NaN)
combined_subset['max_over_p95_ratio'] = np.where(
    (combined_subset['global_p95'] == 0) | (combined_subset['max_value_in_window'].isna()),
    np.nan,
    (combined_subset['max_value_in_window'] - combined_subset['global_p95']) / combined_subset['global_p95']
)

# Format earliest anomaly timestamp as ISO+0800 string, keep NA when missing
combined_subset['earliest_anomaly_timestamp_in_window'] = combined_subset['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
combined_subset.loc[combined_subset['earliest_anomaly_ts'].isna(), 'earliest_anomaly_timestamp_in_window'] = pd.NA

# 4) Prepare the two compact outputs

# A) summary_kpis_df (already prepared). Keep max 20 rows (we have <=9)
summary_kpis_out = summary_kpis_df.head(20)[['cmdb_id','number_of_unique_kpis','top_kpi_examples']].copy()

# B) top-20 (cmdb_id, kpi_name) sorted by max_over_p95_ratio desc
top_kpi_table = combined_subset[
    [
        'cmdb_id',
        'kpi_name',
        'global_p95',
        'points_in_window',
        'points_above_p95_in_window',
        'earliest_anomaly_timestamp_in_window',
        'max_value_in_window',
        'max_over_p95_ratio'
    ]
].copy()

# Sort and take top 20; prefer highest ratio, but only show those with some points in-window ideally
top_kpi_table = top_kpi_table.sort_values(by=['max_over_p95_ratio'], ascending=False).reset_index(drop=True)
top20_kpi_out = top_kpi_table.head(20)

# Keep intermediates in kernel variables for later use:
# subset, kpi_counts, global_stats_subset, window_df, merged_window, window_agg, combined_subset, summary_kpis_out, top20_kpi_out

summary_kpis_out, top20_kpi_out

```
Out[4]:
```
Summary of results for the selected candidates (dockerA1, dockerA2, IG01, MG01, MG02, Tomcat02, Tomcat04, Mysql01, Redis01):

1) Unique KPI counts and examples per host
- Mysql01: 224 unique KPIs (examples: OSLinux-CPU_CPU_CPULoad, many local disk metrics, etc.)
- Redis01: 113 unique KPIs (examples include many network/disk/process KPIs)
- Tomcat04: 109 unique KPIs (examples include Tomcat request/error counts)
- Tomcat02: 109 unique KPIs
- MG02: 93 unique KPIs (includes JVM-Memory_7779 series)
- MG01: 92 unique KPIs
- IG01: 88 unique KPIs
- dockerA1: 6 unique KPIs (container metrics)
- dockerA2: 6 unique KPIs (container metrics)

(Top KPI example lists were truncated for compactness; full lists are in kernel variables.)

2) Top anomalous (cmdb_id, kpi_name) by relative spike (max_over_p95_ratio) — top 20
- The strongest relative spikes (all had 1 point in-window exceeding the global P95) include:
  1. Tomcat04 — OSLinux NETWORK TCP-CLOSE-WAIT: global_p95=0.05, max=1.0, ratio=19.0; earliest exceedance 2021-03-12T14:11:00+08:00.
  2. MG01 — OSLinux NETWORK TCP-FIN-WAIT: global_p95=0.05, max=1.0, ratio=19.0; earliest exceedance 2021-03-12T14:20:00+08:00.
  3. Mysql01 — Mysql-MySQL_3306_Com Insert: global_p95≈0.02667, max≈0.2667, ratio=9.0; earliest exceedance 2021-03-12T14:30:00+08:00.
  4. MG02 — OSLinux NETWORK TCP-CLOSE-WAIT: global_p95=0.15, max=1.0, ratio≈5.67; earliest exceedance 2021-03-12T14:08:00+08:00.
  5. MG02 — OSLinux NETWORK TCP-FIN-WAIT: same pattern as above (ratio≈5.67; earliest 14:16).
  6. Tomcat02 — OSLinux CPU WIO: global_p95≈0.4266, max≈1.8058, ratio≈3.23; earliest exceedance 2021-03-12T14:00:00+08:00.
  7–20. Several disk I/O metrics (DSKRead, DSKRTps, DSKBps) across Mysql01, IG01, MG01, MG02 and local disk on Redis01 with ratios in the ~0.7–2.33 range and earliest exceedances clustered between ~14:05 and ~14:20.

Notes and practical takeaways
- Many of the highest relative spikes are network TCP-state metrics (CLOSE-WAIT, FIN-WAIT) on Tomcat04, MG01 and MG02, and a strong MySQL command spike at the end of the window.
- There are also notable disk I/O and CPU WIO spikes on Tomcat02, Mysql01, IG01, MG01/MG02.
- For these candidate hosts, dockerA1/dockerA2 expose far fewer KPIs than other nodes (only 6 each), so fewer metric signals are available from them.
- All top entries shown had a single in-window point above the global P95 (points_above_p95_in_window = 1); points_in_window vary (≈10–20+ depending on metric).

Recommendation
- Prioritize investigation of: network TCP states on Tomcat04/MG01/MG02, MySQL activity on Mysql01 (Com Insert spike), and disk I/O/CPU WIO on Tomcat02/Mysql01/IG01/MG nodes. Use the stored intermediate tables in the kernel (global_stats_subset, merged_window, window_agg, combined_subset, etc.) for drill-down (timestamps, full series, and raw values).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  number_of_unique_kpis                                   top_kpi_examples
0   Mysql01                    224  OSLinux-CPU_CPU_CPULoad (31), OSLinux-CPU_CPU_...
1   Redis01                    113  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...
2  Tomcat04                    109  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...
3  Tomcat02                    109  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...
4      MG02                     93  JVM-Memory_7779_JVM_Memory_HeapMemoryMax (31),...
5      MG01                     92  JVM-Memory_7779_JVM_Memory_HeapMemoryMax (31),...
6      IG01                     88  JVM-Memory_7778_JVM_Memory_HeapMemoryMax (31),...
7  dockerA1                      6  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...
8  dockerA2                      6  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...,      cmdb_id                                           kpi_name  global_p95  points_in_window  points_above_p95_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  max_over_p95_ratio
0   Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.050000                20                           1             2021-03-12T14:11:00+0800               1.0000           19.000000
1       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.050000                20                           1             2021-03-12T14:20:00+0800               1.0000           19.000000
2    Mysql01                        Mysql-MySQL_3306_Com Insert    0.026670                19                           1             2021-03-12T14:30:00+0800               0.2667            9.000000
3       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.150000                18                           1             2021-03-12T14:08:00+0800               1.0000            5.666667
4       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.150000                18                           1             2021-03-12T14:16:00+0800               1.0000            5.666667
5   Tomcat02                             OSLinux-CPU_CPU_CPUWio    0.426590                18                           1             2021-03-12T14:00:00+0800               1.8058            3.233104
6    Redis01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.030000                15                           1             2021-03-12T14:20:00+0800               0.1000            2.333333
7    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.170010                15                           1             2021-03-12T14:05:00+0800               0.5667            2.333333
8    Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.300000                15                           1             2021-03-12T14:05:00+0800               1.0000            2.333333
9    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    3.159990                15                           1             2021-03-12T14:05:00+0800              10.5333            2.333333
10      IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.039990                15                           1             2021-03-12T14:13:00+0800               0.1333            2.333333
11      IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.005010                15                           1             2021-03-12T14:13:00+0800               0.0167            2.333333
12   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.900000                15                           1             2021-03-12T14:05:00+0800               4.0000            1.105263
13      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.500000                11                           1             2021-03-12T14:18:00+0800               1.0000            1.000000
14      MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.550000                10                           1             2021-03-12T14:17:00+0800               1.0000            0.818182
15      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.018315                10                           1             2021-03-12T14:08:00+0800               0.0333            0.818182
16      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.146685                10                           1             2021-03-12T14:08:00+0800               0.2667            0.818182
17      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    3.359985                10                           1             2021-03-12T14:20:00+0800               6.0000            0.785722
18      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    2.900000                15                           1             2021-03-12T14:17:00+0800               5.0000            0.724138
19  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  110.250000                10                           1             2021-03-12T14:09:00+0800             189.0000            0.714286)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing log_df and tz if available
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure tz-aware timestamp column 'ts' exists
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Focus cmdb_ids and window
focus_cmdbs = ['MG01','MG02','Tomcat02','Tomcat04','IG01','Mysql01','Redis01']
window_start = pd.Timestamp('2021-03-12T14:00:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-12T14:30:00').tz_localize(tz)

# Filter to focus hosts and window (keep for reuse)
log_subset = log_df[log_df['cmdb_id'].isin(focus_cmdbs)].copy()
log_window = log_subset[(log_subset['ts'] >= window_start) & (log_subset['ts'] <= window_end)].copy()

# Ensure numeric 'value' for aggregation where possible
log_window['value_num'] = pd.to_numeric(log_window['value'], errors='coerce')

# 1) Aggregate per (cmdb_id, log_name)
agg = (
    log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        points_in_window=('value', 'size'),
        sum_of_values=('value_num', 'sum'),
        max_value_in_window=('value_num', 'max')
    )
    .reset_index()
)

# Replace sum_of_values NaN with 0 for comparison; keep NaN if all non-numeric? but treat as 0 per requirement
agg['sum_of_values'] = agg['sum_of_values'].fillna(0)
agg['max_value_in_window'] = agg['max_value_in_window'].fillna(0)

# earliest timestamp where value>0 (numeric)
def earliest_ts_positive(g):
    pos = g[g['value_num'] > 0]
    if pos.empty:
        return pd.NA
    return pos['ts'].min().strftime('%Y-%m-%dT%H:%M:%S%z')

earliest = (
    log_window
    .groupby(['cmdb_id','log_name'], dropna=False)
    .apply(earliest_ts_positive)
    .reset_index(name='earliest_timestamp_with_value_gt_0')
)

agg = agg.merge(earliest, on=['cmdb_id','log_name'], how='left')

# Filter rows where sum_of_values>0 or max_value_in_window>0
agg_filtered = agg[(agg['sum_of_values'] > 0) | (agg['max_value_in_window'] > 0)].copy()

# Sort for compact display: by points_in_window desc, then sum_of_values desc
agg_filtered = agg_filtered.sort_values(by=['points_in_window','sum_of_values'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 50 rows for output
agg_out = agg_filtered.head(50)[['cmdb_id','log_name','points_in_window','sum_of_values','max_value_in_window','earliest_timestamp_with_value_gt_0']].copy()

# 2) Detailed rows where log.error_count>0 or log.gc_pause_max>0 within window
detail_filter = log_window[
    ((log_window['log_name'] == 'log.error_count') | (log_window['log_name'] == 'log.gc_pause_max')) &
    (log_window['value_num'].fillna(0) > 0)
].copy()

# Sort by timestamp ascending and limit to 20 rows
detail_rows = detail_filter.sort_values(by='ts', ascending=True).head(20)

# Format detail_rows columns as requested (cmdb_id, timestamp, log_name, value)
detail_rows_out = detail_rows.assign(
    timestamp_iso = detail_rows['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
)[['cmdb_id','timestamp_iso','log_name','value']].rename(columns={'timestamp_iso':'timestamp'})

# Keep intermediate variables in kernel for further use:
# log_subset, log_window, agg, earliest, agg_filtered, agg_out, detail_filter, detail_rows, detail_rows_out

# Return compact outputs (aggregates and detail rows)
agg_out, detail_rows_out

```
Out[5]:
```
Summary (focused on MG01, MG02, Tomcat02, Tomcat04, IG01, Mysql01, Redis01 in 2021-03-12T14:00–14:30+08:00):

1) Aggregated log metrics (only rows with positive sums or max shown; top results)
- Tomcat02 — log.row_count: 31 points in window, sum = 13,761, max = 574, earliest positive = 2021-03-12T14:00:00+08:00
- Tomcat04 — log.row_count: 31 points in window, sum = 13,612, max = 543, earliest positive = 2021-03-12T14:00:00+08:00
- Tomcat02 — log.gc_pause_max: 30 points in window, sum = 2.20 (seconds), max = 0.10, earliest positive = 2021-03-12T14:00:00+08:00
- Mysql01 — log.row_count: 28 points in window, sum = 159, max = 16, earliest positive = 2021-03-12T14:00:00+08:00
- Tomcat04 — log.gc_pause_max: 23 points in window, sum = 1.47 (seconds), max = 0.11, earliest positive = 2021-03-12T14:00:00+08:00

(Other focus hosts had no log aggregates with sum>0 or max>0 in this filtered output; results limited to top rows.)

2) Detailed rows (events where log.error_count>0 or log.gc_pause_max>0)
- Returned up to 20 detail rows (sorted by timestamp). All returned events are log.gc_pause_max > 0 on Tomcat02 and Tomcat04.
- GC pause events appear every minute starting at 2021-03-12T14:00:00+08:00, with small per-minute pauses (examples):
  - 2021-03-12T14:00:00+08:00 — Tomcat02: gc_pause_max = 0.06
  - 2021-03-12T14:00:00+08:00 — Tomcat04: gc_pause_max = 0.07
  - (repeated per-minute entries for both Tomcat02 and Tomcat04 through ~14:11:00+08:00 in the returned slice)
- No log.error_count > 0 entries for the selected hosts in the window (none returned).

Quick interpretation / next steps
- Tomcat02 and Tomcat04 show sustained per-minute GC pause activity during the window (small pauses but recurring) and very high log.row_count totals — these are primary candidates to inspect further (heap/GC tuning, traffic surge).
- Mysql01 shows modest log.row_count spikes (max 16) and should be checked if correlated with application load peaks.
- No explicit error_count events were observed for these hosts during the incident window; focus on GC and request-load diagnostics first.

Full intermediate tables (aggregates and detail rows) remain available in kernel variables for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  points_in_window  sum_of_values  max_value_in_window earliest_timestamp_with_value_gt_0
0  Tomcat02     log.row_count                31       13761.00               574.00           2021-03-12T14:00:00+0800
1  Tomcat04     log.row_count                31       13612.00               543.00           2021-03-12T14:00:00+0800
2  Tomcat02  log.gc_pause_max                30           2.20                 0.10           2021-03-12T14:00:00+0800
3   Mysql01     log.row_count                28         159.00                16.00           2021-03-12T14:00:00+0800
4  Tomcat04  log.gc_pause_max                23           1.47                 0.11           2021-03-12T14:00:00+0800,       cmdb_id                 timestamp          log_name  value
6    Tomcat02  2021-03-12T14:00:00+0800  log.gc_pause_max   0.06
11   Tomcat04  2021-03-12T14:00:00+0800  log.gc_pause_max   0.07
23   Tomcat02  2021-03-12T14:01:00+0800  log.gc_pause_max   0.06
29   Tomcat04  2021-03-12T14:01:00+0800  log.gc_pause_max   0.07
40   Tomcat02  2021-03-12T14:02:00+0800  log.gc_pause_max   0.07
46   Tomcat04  2021-03-12T14:02:00+0800  log.gc_pause_max   0.05
58   Tomcat02  2021-03-12T14:03:00+0800  log.gc_pause_max   0.06
74   Tomcat02  2021-03-12T14:04:00+0800  log.gc_pause_max   0.06
80   Tomcat04  2021-03-12T14:04:00+0800  log.gc_pause_max   0.06
89   Tomcat02  2021-03-12T14:05:00+0800  log.gc_pause_max   0.08
95   Tomcat04  2021-03-12T14:05:00+0800  log.gc_pause_max   0.06
107  Tomcat02  2021-03-12T14:06:00+0800  log.gc_pause_max   0.07
130  Tomcat04  2021-03-12T14:07:00+0800  log.gc_pause_max   0.07
124  Tomcat02  2021-03-12T14:07:00+0800  log.gc_pause_max   0.10
140  Tomcat02  2021-03-12T14:08:00+0800  log.gc_pause_max   0.06
145  Tomcat04  2021-03-12T14:08:00+0800  log.gc_pause_max   0.06
156  Tomcat02  2021-03-12T14:09:00+0800  log.gc_pause_max   0.07
162  Tomcat04  2021-03-12T14:09:00+0800  log.gc_pause_max   0.08
174  Tomcat02  2021-03-12T14:10:00+0800  log.gc_pause_max   0.09
191  Tomcat02  2021-03-12T14:11:00+0800  log.gc_pause_max   0.05)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables where possible
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure tz-aware timestamp column 'ts' exists
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Use previously computed global P95s if available
# Prefer global_stats (full-series P95 computed earlier). Fallback to global_stats_subset.
if 'global_stats' in globals():
    p95_table = global_stats.copy()
elif 'global_stats_subset' in globals():
    p95_table = global_stats_subset.copy()
else:
    raise RuntimeError("No precomputed global P95 table found in kernel (expected 'global_stats' or 'global_stats_subset').")

# Focus on MG01 and MG02 and the tight window
targets = ['MG01', 'MG02']
window_start = pd.Timestamp('2021-03-12T14:20:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-12T14:26:00').tz_localize(tz)

# Slice metric data for targets and window (keep full-series p95 in p95_table)
window_slice = metric_df[
    (metric_df['cmdb_id'].isin(targets)) &
    (metric_df['ts'] >= window_start) &
    (metric_df['ts'] <= window_end)
].copy()

# Merge global p95 onto the windowed rows (do not recompute thresholds)
merged_window_mg = window_slice.merge(
    p95_table[['cmdb_id', 'kpi_name', 'global_p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left',
    validate='m:1'
)

# Drop rows where global_p95 is missing (can't evaluate without global threshold)
merged_window_mg = merged_window_mg[~merged_window_mg['global_p95'].isna()].copy()

# Flag values above the precomputed global P95 (strictly greater)
merged_window_mg['above_p95'] = merged_window_mg['value'] > merged_window_mg['global_p95']

# Prepare anomaly timestamp where above_p95
merged_window_mg['anom_ts'] = merged_window_mg['ts'].where(merged_window_mg['above_p95'], pd.NaT)

# Aggregate per (cmdb_id, kpi_name) within the tight window
agg_mg = (
    merged_window_mg
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        points_in_range=('value', 'size'),
        points_above_p95_in_range=('above_p95', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),
        max_value_in_range=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    .reset_index()
)

# Keep only KPIs that have at least one value > global_p95 in this window
agg_mg = agg_mg[agg_mg['points_above_p95_in_range'] > 0].copy()

# Compute max_over_p95_ratio safely (NaN if global_p95 == 0)
agg_mg['max_over_p95_ratio'] = np.where(
    agg_mg['global_p95'] == 0,
    np.nan,
    (agg_mg['max_value_in_range'] - agg_mg['global_p95']) / agg_mg['global_p95']
)

# Format earliest anomaly timestamp as ISO+0800 string; keep NA when missing
agg_mg['earliest_anomaly_timestamp_in_range'] = agg_mg['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
agg_mg.loc[agg_mg['earliest_anomaly_ts'].isna(), 'earliest_anomaly_timestamp_in_range'] = pd.NA

# Select and order requested columns
result_mg = agg_mg[
    [
        'cmdb_id',
        'kpi_name',
        'global_p95',
        'points_in_range',
        'points_above_p95_in_range',
        'earliest_anomaly_timestamp_in_range',
        'max_value_in_range',
        'max_over_p95_ratio'
    ]
].copy()

# Sort by max_over_p95_ratio desc and limit to 50 rows (compact output)
result_mg = result_mg.sort_values(by='max_over_p95_ratio', ascending=False).reset_index(drop=True)
result_mg_head50 = result_mg.head(50)

# Keep intermediate variables for later use: window_slice, merged_window_mg, agg_mg, result_mg
result_mg_head50

```
Out[6]:
```
Summary of MG01 / MG02 metrics in the tight window 2021-03-12T14:20–14:26+08:00 (using precomputed global P95s):

Top signals (only metrics that exceeded their global P95 at least once in this window)
- MG01 — OSLinux NETWORK TCP-FIN-WAIT
  - global_p95 = 0.05, points_in_range = 5, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max value = 1.0 → max_over_p95_ratio = 19.0 (largest relative spike)

- MG01 — sda_DSKRead (local disk read)
  - global_p95 ≈ 3.36, points_in_range = 2, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max value = 6.0 → ratio ≈ 0.786

- MG01 — sda_DSKRTps (disk requests/sec)
  - global_p95 ≈ 0.1083, points_in_range = 2, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max value ≈ 0.1833 → ratio ≈ 0.692

- MG02 — sda_DSKBps (disk throughput)
  - global_p95 = 1.5, points_in_range = 3, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max value = 2.0 → ratio = 0.333

- MG02 — sda_DSKWTps (disk write TPS)
  - global_p95 ≈ 1.1467, points_in_range = 3, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max value ≈ 1.2667 → ratio ≈ 0.105

- MG02 — ens160_NETKBTotalPerSec (network KB/sec)
  - global_p95 ≈ 795,368.5, points_in_range = 3, 1 point > P95
  - earliest exceedance: 2021-03-12T14:20:00+08:00
  - max ≈ 874,160 → ratio ≈ 0.099

- MG02 — single CPU/core util and other CPU/JVM metrics
  - small relative increases (ratios ≈ 0.0015–0.069). earliest exceedances around 14:21–14:25.

- Additional MG01/MG02 disk/network metrics also show single-point exceedances with ratios in the ~0.001–0.8 range, earliest mostly at 14:20.

Interpretation and recommended next steps
- A short-lived spike centered at ~14:20:00+08:00 affected MG01 and MG02: strongest anomalies are network TCP state (FIN-WAIT) and disk read/IO metrics, plus minor CPU/JVM upticks.
- Most anomalies are single-minute spikes (points_above_p95_in_range = 1 for each KPI), except some metrics had multiple points in the slice but still only one exceedance.
- Priority actions: investigate network stack and socket states on MG01 (FIN-WAIT), check disk subsystem performance (sda reads/IO) on both MG01 and MG02 at ~14:20, and correlate with trace-level spike timestamps and any traffic bursts or deployments at that time.
- Full intermediate tables (per-timestamp rows, merged P95s, and aggregated results) remain in kernel variables for deeper drilling.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name     global_p95  points_in_range  points_above_p95_in_range earliest_anomaly_timestamp_in_range  max_value_in_range  max_over_p95_ratio
0     MG01      OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT       0.050000                5                          1            2021-03-12T14:20:00+0800              1.0000           19.000000
1     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead       3.359985                2                          1            2021-03-12T14:20:00+0800              6.0000            0.785722
2     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps       0.108330                2                          1            2021-03-12T14:20:00+0800              0.1833            0.692052
3     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps       1.500000                3                          1            2021-03-12T14:20:00+0800              2.0000            0.333333
4     MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps       1.146685                3                          1            2021-03-12T14:20:00+0800              1.2667            0.104663
5     MG02   OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  795368.500000                3                          1            2021-03-12T14:20:00+0800         874160.0000            0.099063
6     MG02                   OSLinux-CPU_CPU-0_SingleCpuUtil       6.403230                3                          1            2021-03-12T14:25:00+0800              6.8454            0.069054
7     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite      15.060000                3                          1            2021-03-12T14:20:00+0800             15.6000            0.035857
8     MG02         JVM-Operating System_7779_JVM_JVM_CPULoad       2.624920                5                          1            2021-03-12T14:20:00+0800              2.7160            0.034698
9     MG02                        OSLinux-CPU_CPU_CPUSysTime       1.266850                4                          1            2021-03-12T14:21:00+0800              1.2817            0.011722
10    MG02      OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut    1290.000000                3                          1            2021-03-12T14:20:00+0800           1293.0000            0.002326
11    MG02                       OSLinux-CPU_CPU_CPUUserTime      26.666810                4                          1            2021-03-12T14:21:00+0800             26.7083            0.001556
12    MG02                        OSLinux-CPU_CPU_CPUCpuUtil      28.744985                5                          1            2021-03-12T14:25:00+0800             28.7884            0.001510
13    MG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn    1374.000000                3                          1            2021-03-12T14:20:00+0800           1376.0000            0.001456
14    MG01                   OSLinux-CPU_CPU-3_SingleCpuidle      97.498780                2                          1            2021-03-12T14:22:00+0800             97.6373            0.001421```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df and tz if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure tz-aware datetime column 'ts' exists
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Use precomputed global P95s if available; otherwise compute from full trace_df (global before filtering)
if 'global_stats_trace' in globals():
    p95_table = global_stats_trace[['cmdb_id', 'trace_name', 'global_p95']].copy()
else:
    p95_table = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# Filter trace_df to targets and the requested tight interval (inclusive)
targets = ['dockerA1','dockerA2','MG01','MG02']
start = pd.Timestamp('2021-03-12T14:16:00').tz_localize(tz)
end = pd.Timestamp('2021-03-12T14:25:00').tz_localize(tz)

trace_slice = trace_df[
    (trace_df['cmdb_id'].isin(targets)) &
    (trace_df['ts'] >= start) &
    (trace_df['ts'] <= end)
].copy()

# Merge global_p95 (precomputed) onto the slice
merged_trace_slice = trace_slice.merge(p95_table, on=['cmdb_id','trace_name'], how='left')

# Drop rows without a global_p95 (can't evaluate)
merged_trace_slice = merged_trace_slice[~merged_trace_slice['global_p95'].isna()].copy()

# Compute per-row above_p95 and ratio (safe divide)
merged_trace_slice['above_p95'] = merged_trace_slice['value'] > merged_trace_slice['global_p95']
merged_trace_slice['over_p95_ratio'] = np.where(
    merged_trace_slice['global_p95'] == 0,
    np.nan,
    (merged_trace_slice['value'] - merged_trace_slice['global_p95']) / merged_trace_slice['global_p95']
)

# Keep only entries where value > global_p95
trace_anomalies = merged_trace_slice[merged_trace_slice['above_p95']].copy()

# Format timestamp as ISO+0800 string for display
trace_anomalies['timestamp'] = trace_anomalies['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# Sort by timestamp ascending, then by over_p95_ratio descending
trace_anomalies = trace_anomalies.sort_values(by=['ts','over_p95_ratio'], ascending=[True, False])

# Select requested columns and limit to max 50 rows
trace_anomalies_slice = trace_anomalies[
    ['cmdb_id','trace_name','timestamp','value','global_p95','over_p95_ratio']
].head(50).reset_index(drop=True)

# Keep intermediate variables in kernel for further analysis:
# trace_slice, merged_trace_slice, trace_anomalies, trace_anomalies_slice

trace_anomalies_slice

```
Out[7]:
```
summary = (
    "Findings for traces in 2021-03-12T14:16:00+08:00 through 2021-03-12T14:25:00+08:00 (only rows where value > precomputed global P95):\n\n"
    "1) First exceedances (earliest timestamp = 2021-03-12T14:16:00+08:00):\n"
    "   - MG02 and dockerA nodes showed the very first breaches at 14:16:00+08:00.\n"
    "     Examples: MG02 trace.to_dockerA1.row_count = 209 (> global_p95 200.5, +4.2%),\n"
    "               dockerA1 trace.from_MG02.row_count = 209 (>200.5, +4.2%),\n"
    "               MG02 trace.from_Tomcat02.row_count = 292 (>283.75, +2.9%).\n"
    "   - These early breaches are primarily row_count (traffic) increases and are modest in relative size (a few percent over P95).\n\n"
    "2) Follow-up/broader exceedances around 14:20:00+08:00:\n"
    "   - At ~14:20 many MG01/MG02 trace duration metrics exceeded their P95s (latency spikes).\n"
    "     Examples: MG01 trace.self.duration_mean = 0.45396 (>0.43855, +3.5%),\n"
    "               MG01 trace.from_Tomcat04.duration_mean = 0.60704 (>0.58939, +3.0%),\n"
    "               MG02 trace.to_dockerB2.duration_mean = 0.68032 (>0.66030, +3.0%),\n"
    "               MG01 trace.self.duration_p95 = 2.1489 (>2.1260, +1.1%).\n"
    "   - These indicate a clear latency/duration increase at ~14:20 affecting MG nodes and their paths to dockerA/dockerB/Tomcat.\n\n"
    "Conclusion / recommendation:\n"
    " - The incident timeline suggests an initial traffic/row_count increase at 14:16 (MG02 ↔ dockerA), followed by broader latency spikes on MG01/MG02 around 14:20.\n"
    " - Investigate MG02 and dockerA routing/traffic patterns at ~14:16, then investigate MG node latency, I/O and downstream Tomcat interactions around ~14:20.\n"
    " - Correlate these trace exceedances with the metric-level anomalies (network, disk, and JVM signals) already observed at the same timestamps."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name                 timestamp       value  global_p95  over_p95_ratio
0       MG02        trace.to_dockerA1.row_count  2021-03-12T14:16:00+0800  209.000000  200.500000        0.042394
1   dockerA1          trace.from_MG02.row_count  2021-03-12T14:16:00+0800  209.000000  200.500000        0.042394
2       MG02      trace.from_Tomcat02.row_count  2021-03-12T14:16:00+0800  292.000000  283.750000        0.029075
3       MG02        trace.to_dockerA2.row_count  2021-03-12T14:16:00+0800  208.000000  207.750000        0.001203
4   dockerA2          trace.from_MG02.row_count  2021-03-12T14:16:00+0800  208.000000  207.750000        0.001203
..       ...                                ...                       ...         ...         ...             ...
45      MG01           trace.self.duration_mean  2021-03-12T14:20:00+0800    0.453960    0.438552        0.035134
46      MG02    trace.to_dockerB2.duration_mean  2021-03-12T14:20:00+0800    0.680319    0.660298        0.030321
47      MG01  trace.from_Tomcat04.duration_mean  2021-03-12T14:20:00+0800    0.607036    0.589394        0.029932
48      MG02  trace.from_dockerB2.duration_mean  2021-03-12T14:20:00+0800    0.154247    0.151313        0.019392
49      MG01            trace.self.duration_p95  2021-03-12T14:20:00+0800    2.148900    2.126040        0.010752

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```